# IndiVoice-DeepASR: Resuming Training (Step 1200 -> 2000)

**Status**: Resuming from `purvansh01/whisper-indian-lora` (Step 1200).

### How this works:
1. **Clones Code**: Gets the latest training logic.
2. **Pulls Weights**: Downloads the Step 1200 adapter from Hugging Face Hub.
3. **Resumes**: The `Seq2SeqTrainer` detects the local checkpoint and continues until Step 2,000.

In [ ]:
# 1. Setup Environment & Source Code
import os, shutil

# Injection point for HF_TOKEN (Use Kaggle Secrets for safety)
os.environ['HF_TOKEN'] = os.environ.get('HF_TOKEN', 'YOUR_HF_TOKEN_HERE')

repo_dir = '/kaggle/working/IndiVoice-DeepASR'
if os.path.exists(repo_dir):
    print('[LOG] Cleaning environment...')
    shutil.rmtree(repo_dir)

print('[LOG] Cloning Repository...')
!git clone https://github.com/purvanshjoshi/IndiVoice-DeepASR.git {repo_dir}

%cd {repo_dir}

print('[LOG] Running Infrastructure Setup...')
!bash kaggle/setup_kaggle.sh

print('[LOG] Setup Complete.')

In [ ]:
# 2. Recover Checkpoint from Hugging Face
from huggingface_hub import snapshot_download
import glob

model_id = "purvansh01/whisper-indian-lora"
output_dir = "/kaggle/working/models/whisper-indian-lora"

print(f"[LOG] Downloading latest checkpoint from {model_id}...")

try:
    snapshot_download(
        repo_id=model_id,
        local_dir=output_dir,
        token=os.environ['HF_TOKEN'],
        library_name="transformers"
    )
    
    # Verify checkpoint detection
    checkpoints = glob.glob(f"{output_dir}/checkpoint-*")
    if checkpoints:
        print(f"[SUCCESS] Recovered checkpoints: {checkpoints}")
    else:
        print("[WARNING] No 'checkpoint-*' folders found. Check your Hugging Face repo structure.")
        
except Exception as e:
    print(f"[ERROR] Failed to download checkpoint: {e}")

In [ ]:
# 3. Launch Resumed Training
!accelerate launch src/train.py \
    --model_name openai/whisper-medium \
    --output_dir /kaggle/working/models/whisper-indian-lora \
    --batch_size 8 \
    --grad_accum 2 \
    --epochs 3